## Load and prepare data 

Load and prepare data for backtrajectory calculations.

- Load the NSIDC PP data: `ds`

    - Loading a single, concatenated dataset from multiple source files.
    - Interpolating over a few holes in the data.
    
- Load a land mask: `MASK`
    - Define a land mask of land/no land that we use to kill trajectories that intersect a coastline.

In [1]:
import numpy as np
import xarray as xr
import glob
import pyproj
import matplotlib.pyplot as plt
from matplotlib.dates import date2num
import cmocean
import re

### Load data

Replace the *ddir* folder with the directory containing the PP NSIDC files (e.g. *icemotion_daily_nh_25km_20120101_20121231_v4.1.nc*)

Set year bounds; exclude years you are sure not to use (we will load the whole thing in memory, so good to be mindful of size - but 2014-2021 has been totally fine for me).

In [4]:
ddir ='/media/oyvindl/ratatosk/data/sea_ice_drift/nsidc_polar_pathfinder/modern_subset/'
ddir ='/media/oyvindf/SKADI/sea_ice/sea_ice_drift/nsidc_polar_pathfinder_v4/source/annual_drift_vectors/'

flist = glob.glob(ddir + '*nc')


# Can reduce the dataset, not loading everything
if True:
    min_year = 2014
    max_year = 2021
    
    flist = [
        f for f in flist
        if min_year <= int(re.search(r'_(\d{4})\d{4}_', f).group(1)) <= max_year
    ]

Lazy load of the data (concatenating the files) 

In [28]:
# Loading
# ("minimal" keyword avoids adding a time dimension to lat,lon)
# chunks/parallel: trying to optimize memory usage..
try:
    ds = xr.open_mfdataset(
          flist, data_vars = 'minimal', 
          chunks = {'time': 50, 'y': 'auto', 'x': 'auto'},
          parallel = True)
except:
    raise Exception('Failed to open files..\n(Looked for files here: %s)'%ddir)
# Last value is empty -> drop
ds = ds.drop_sel(time=ds.time.data[-1])

### Interpolate over NaN days

Some days, like 31.12 on 2020 and 2021, are all nans. 

>Update 14.01: Not any more??

-> Linearly interpolating between adjectent days (considering NaNs as zeros, 
but setting all-zero points to nan..)

In [37]:
# Step 1: Find time slices where the whole u field is NaN
isnan_all = ds.u.max(dim=['x','y']).isnull()
is_nan_indices = np.where(isnan_all.compute())[0]  
# triggers dask computation just once

# Step 2: Shift u and v along time
u_prev = ds.u.shift(time=1)
u_next = ds.u.shift(time=-1)
v_prev = ds.v.shift(time=1)
v_next = ds.v.shift(time=-1)

# Step 3: Interpolate only where previous and next slices exist
u_interp = 0.5 * (u_prev + u_next)
v_interp = 0.5 * (v_prev + v_next)

# Step 4: Replace only fully missing slices
ds['u'] = xr.where(isnan_all, u_interp, ds.u)
ds['v'] = xr.where(isnan_all, v_interp, ds.v)


In [47]:
ds0 = ds.sel(time = '2020-12-31').squeeze()

##### (Optional): Quick check

Toggle *check_interpolation=True* to show a quick check of the interpolation on Dec 31 2020..


In [5]:
check_interpolation = False

if check_interpolation:
    x_point, y_point = 180, 180
    
    fig, ax = plt.subplots(figsize = (11, 3))
    
    ax.plot_date(date2num(ds.time), ds.u.isel(x=x_point, y=y_point), '-k', alpha = 0.3)
    ax.plot_date(date2num(ds.time), ds.u.isel(x=x_point, y=y_point), '.', ms = 6, 
                 label = 'u (original)')
    ax.plot_date(ds.time[is_nan_indices[0]], 
                 ds.u.isel(x=x_point, y=y_point)[is_nan_indices[0]], '*y', ms = 12,
                 label = 'u (interpolated)')
    ax.plot_date(date2num(ds.time), ds.v.isel(x=x_point, y=y_point), '-k', alpha = 0.3)
    ax.plot_date(date2num(ds.time), ds.v.isel(x=x_point, y=y_point), '.', ms = 6, 
                 label = 'v (original)')
    ax.plot_date(ds.time[is_nan_indices[0]], 
                 ds.v.isel(x=x_point, y=y_point)[is_nan_indices[0]], '*r', ms = 12, 
                 label = 'v (interpolated)')

    for t_ in date2num(ds.time):
        ax.vlines(t_, -50, 50, 'k', alpha = 0.2)
    ax.set_xlim(18618, 18633)
    ax.set_ylim(-10, 10)
    ax.legend(ncol = 2, fontsize = 9)

## Deprecated checks

Not sure if this works anymore.

In [6]:
import matplotlib.pyplot as plt
from matplotlib.dates import date2num, num2date

check_interpolation = False

if check_interpolation:
    x_point, y_point = 180, 180  # grid point to inspect
    
    # Convert cftime to matplotlib numeric dates
    times_num = date2num(ds.time.values)
    
    # Get u and v time series at the selected grid point
    u_vals = ds.u.isel(x=x_point, y=y_point).values
    v_vals = ds.v.isel(x=x_point, y=y_point).values
    
    # Setup figure
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # Plot original time series as lines
    ax.plot(times_num, u_vals, '-k', alpha=0.3, label='u (original)')
    ax.plot(times_num, v_vals, '-k', alpha=0.3, label='v (original)')
    
    # Scatter all points for clarity
    ax.scatter(times_num, u_vals, color='blue', s=20, label='u (all points)')
    ax.scatter(times_num, v_vals, color='green', s=20, label='v (all points)')
    
    # Highlight interpolated points
    ax.scatter(times_num[is_nan_indices], u_vals[is_nan_indices], color='orange', s=60, marker='*', label='u (interpolated)')
    ax.scatter(times_num[is_nan_indices], v_vals[is_nan_indices], color='red', s=60, marker='*', label='v (interpolated)')
    
    # Optional: vertical lines for each time step
    for t_ in times_num:
        ax.axvline(t_, color='k', alpha=0.05)
    
    ax.set_ylim(-10, 10)
    ax.legend(ncol=2, fontsize=9)
    ax.set_title(f"Interpolation check at grid point ({x_point}, {y_point})")
    
    # Format x-axis as dates
    import matplotlib.dates as mdates
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    fig.autofmt_xdate()
    
    plt.show()


### Define a land mask

Here, we can define a land mask (*MASK*) that will be used to kill a backtrajectory when it intersects a coastline.

*MASK* should have the following variables:
- *x, y* - Grid 
- *is_land* - Boolean (land=1, not land = 0)  
- *proj* - pyproj.CRS defining the grid projection of *x, y*


In my case, I am using a 2km land mask for the Eurasian sector based on IBCAO-v4, where I have removed Svalbard in order to allow trajectories to pass through islands. This mask was made elsewhere - I am just loading it here.

We can choose not to use a land mask. In that case, set *no_land_mask = True*

In [7]:
no_land_mask = False

In [8]:
# Load the land mask based on IBCAO (and defining the projection of this mask):
if not no_land_mask:
    my_mask_proj = pyproj.CRS('epsg:3996')  
    my_mask = xr.open_dataset(
        '/media/oyvindf/SKADI/geodata/bathymetry/ibcao_v4/netcdf/IBCAO_landmask_greater_barents_2km.nc')
    mask_is_land = my_mask.landmask_wo_svb
    # (Also adding lon/lat, but thise aren't strictly necessary)
    MASK = xr.Dataset(coords = {'x':my_mask.x, 'y':my_mask.y},
                     data_vars = {'lon':my_mask.lon, 'lat':my_mask.lat,
                                  'is_land':mask_is_land})
    MASK['proj'] = ((), my_mask_proj)

In [9]:
# If we dont have a land mask/dont want to use one:
if no_land_mask:
    MASK = None

##### (Optional): Quick check of mask

Toggle *show_mask=True* to show a quick landmask plot

In [10]:
show_mask = False

if show_mask:

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    lat_ctr, lon_ctr = 86, 30

    cart_proj = ccrs.Stereographic(
        central_latitude=lat_ctr,
        central_longitude=lon_ctr,
        scale_factor=None, globe=None)

    fig = plt.figure(figsize = (8, 8))
    ax = plt.axes(projection=cart_proj)

    ax.set_facecolor('gray')
    # Coastline
    ax.coastlines('50m', color = 'w', lw = 0.5 )

    # Land
    land_50m = cfeature.NaturalEarthFeature(
        'physical', 'land', '50m',
        edgecolor='k', linewidth = 1,
        facecolor=cfeature.COLORS['land'])

   # ax.add_feature(land_50m)

    # Set boundaries
    ax.set_extent([-20, 191, 66, 89.5], crs=ccrs.PlateCarree())
    
    # Transform mask coordiates (x-y) to (lat-lon) 
  #  wgs84_proj = pyproj.CRS("EPSG:4326") # latlon with WGS84 datum
  #  MASK_to_latlon = pyproj.Transformer.from_crs(MASK.proj.data.item(), wgs84_proj)
  #  mask_lon, mask_lat = None, None

    ax.pcolormesh(MASK.x, MASK.y, MASK.is_land, 
                  cmap = cmocean.cm.haline, transform = ccrs.epsg(3996))
    

In [12]:
print(my_mask)

<xarray.Dataset> Size: 3MB
Dimensions:          (x: 240, y: 440)
Coordinates:
  * x                (x) float64 2kB -1.058e+05 -9.58e+04 ... 2.284e+06
  * y                (y) float64 4kB -1.706e+06 -1.696e+06 ... 2.684e+06
Data variables:
    lon              (y, x) float32 422kB ...
    lat              (y, x) float32 422kB ...
    landmask         (y, x) float64 845kB ...
    landmask_wo_svb  (y, x) float64 845kB ...
Attributes:
    Conventions:  COARDS, CF-1.5
    title:        Produced by grdconvert
    history:      grdconvert cut/ibcao-v4.2-400m.tiff cut/ibcao-v4.2-400m.nc
    description:  
    GMT_version:  5.4.5 [64-bit]
    node_offset:  1
